                                                                Basic Statistics-2 

In [1]:
import pandas as pd

In [5]:
patients_df = pd.read_csv("Patient_Data.csv")
billing_df = pd.read_csv("Billing_Data.csv")
patient_df.head()

,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime
0,101,Alice,Cardiology,Dr. Smith,5000.0,1,2023-01-10 09:00
1,102,Bob,Neurology,Dr. John,NaN,2,2023-01-11 10:30
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,1,2023-01-12 11:00
3,104,David,Cardiology,Dr. Smith,6200.0,3,2023-01-13 12:00
4,105,Eva,Dermatology,Dr. Rose,NaN,2,2023-01-14 08:45


In [9]:
#Show dataset summary
print("Patient Dataset Info:")
patients_df.info()

Patient Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   PatientID   6 non-null      int64  
 1   Department  6 non-null      object 
 2   Doctor      6 non-null      object 
 3   BillAmount  4 non-null      float64
dtypes: float64(1), int64(1), object(2)
memory usage: 324.0+ bytes


In [15]:
#Select relevant billing columns
patients_df.columns = patients_df.columns.str.strip()
billing_cols = ['PatientID', 'Department', 'Doctor', 'BillAmount']
patients_df = patients_df[[col for col in billing_cols if col in patients_df.columns]]

In [17]:
#Drop administrative columns
patients_df = patients_df.drop(columns=['ReceptionistID', 'CheckInTime'], errors='ignore')
patient_df.head()

,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime
0,101,Alice,Cardiology,Dr. Smith,5000.0,1,2023-01-10 09:00
1,102,Bob,Neurology,Dr. John,NaN,2,2023-01-11 10:30
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,1,2023-01-12 11:00
3,104,David,Cardiology,Dr. Smith,6200.0,3,2023-01-13 12:00
4,105,Eva,Dermatology,Dr. Rose,NaN,2,2023-01-14 08:45


In [18]:
#Fill missing BillAmount with mean
mean_bill=patients_df['BillAmount'].mean()
patients_df['BillAmount']=patients_df['BillAmount'].fillna(mean_bill)

In [21]:
# Remove duplicate patient records
patients_df=patients_df.drop_duplicates(subset='PatientID')
patient_df.head()

,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime
0,101,Alice,Cardiology,Dr. Smith,5000.0,1,2023-01-10 09:00
1,102,Bob,Neurology,Dr. John,NaN,2,2023-01-11 10:30
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,1,2023-01-12 11:00
3,104,David,Cardiology,Dr. Smith,6200.0,3,2023-01-13 12:00
4,105,Eva,Dermatology,Dr. Rose,NaN,2,2023-01-14 08:45


In [23]:
#Group by department (total revenue)
dept_revenue=patients_df.groupby('Department')['BillAmount'].sum()
print("\nDepartment-wise Total Revenue:")
print(dept_revenue)


Department-wise Total Revenue:
Department
Cardiology     11200.0
Dermatology     5925.0
Neurology       5925.0
Orthopedics     7500.0
Name: BillAmount, dtype: float64


In [24]:
#Merge with billing dataset
merged_df=pd.merge(patients_df,billing_df,on='PatientID',how='inner')

In [26]:
#Row-wise concatenation
new_patients=pd.DataFrame({
    'PatientID':[201, 202],
    'Department':['Cardiology','Neurology'],
    'Doctor':['Dr.Smith','Dr.Lee'],
    'BillAmount':[5000,7000]})
patient_df.head()

,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime
0,101,Alice,Cardiology,Dr. Smith,5000.0,1,2023-01-10 09:00
1,102,Bob,Neurology,Dr. John,NaN,2,2023-01-11 10:30
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,1,2023-01-12 11:00
3,104,David,Cardiology,Dr. Smith,6200.0,3,2023-01-13 12:00
4,105,Eva,Dermatology,Dr. Rose,NaN,2,2023-01-14 08:45


In [27]:
# Add to merged dataset
merged_df=pd.concat([merged_df,new_patients],axis=0,ignore_index=True)

In [30]:
#Column-wise concatenation
new_columns = pd.DataFrame({
    'InsuranceCovered': ([True, False] * len(merged_df))[:len(merged_df)],
    'FinalAmount': merged_df['BillAmount'] * 0.9
})
patient_df.head()

,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime
0,101,Alice,Cardiology,Dr. Smith,5000.0,1,2023-01-10 09:00
1,102,Bob,Neurology,Dr. John,NaN,2,2023-01-11 10:30
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,1,2023-01-12 11:00
3,104,David,Cardiology,Dr. Smith,6200.0,3,2023-01-13 12:00
4,105,Eva,Dermatology,Dr. Rose,NaN,2,2023-01-14 08:45


In [32]:
new_columns=new_columns.iloc[:len(merged_df)]

In [33]:
#new columns
final_df=pd.concat([merged_df,new_columns],axis=1)

In [34]:
print("\nFinal Cleaned Dataset:")
print(final_df.head())

final_df.to_csv("final_hospital_data.csv",index=False)


Final Cleaned Dataset:
   PatientID   Department     Doctor  BillAmount  InsuranceCovered  \
0        101   Cardiology  Dr. Smith      5000.0            2000.0   
1        102    Neurology   Dr. John      5925.0            1500.0   
2        103  Orthopedics    Dr. Lee      7500.0            2500.0   
3        104   Cardiology  Dr. Smith      6200.0            3000.0   
4        105  Dermatology   Dr. Rose      5925.0            1000.0   

   FinalAmount  InsuranceCovered  FinalAmount  
0       3000.0              True       4500.0  
1       3500.0             False       5332.5  
2       5000.0              True       6750.0  
3       3200.0             False       5580.0  
4       4000.0              True       5332.5  
